In [0]:
# Install dependencies for PDF processing
# - pypdf: For reading and extracting text from PDF files
# - pandas: For data manipulation and DataFrame creation
%pip install pypdf pandas

# Restart Python to load newly installed packages
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# ============================================================================
# PDF INGESTION FROM UNITY CATALOG VOLUME
# ============================================================================
import os
import pandas as pd
from pypdf import PdfReader

# Define the Volume path containing COVID-19 PDF documents
volume_path = "/Volumes/main/covid_rag/covid_docs/"

# List all PDF files in the Volume
files = [f.path for f in dbutils.fs.ls(volume_path) if f.path.endswith(".pdf")]

# Extract text from each page of each PDF
rows = []
for file_path in files:
    # Convert DBFS path to local path for PyPDF
    local_path = file_path.replace("dbfs:", "")
    reader = PdfReader(local_path)
    
    # Iterate through all pages in the PDF
    for page_num, page in enumerate(reader.pages, start=1):
        # Extract text from the page
        text = page.extract_text() or ""
        
        # Store page metadata and text
        rows.append({
            "source_file": os.path.basename(file_path),  # Just the filename
            "page_num": page_num,
            "text": text.strip(),
            "source_path": file_path  # Full path for reference
        })

# Create DataFrame from extracted pages
df_pages = pd.DataFrame(rows)

# Filter out pages with very little text (likely empty or image-only pages)
# Keep only pages with more than 50 characters
df_pages = df_pages[df_pages["text"].str.len() > 50]

# Display sample of extracted pages
display(df_pages.head())

source_file,page_num,text,source_path
Doc_1_COVID-19_Overview.pdf,1,"1 Coronavirus disease (COVID-19) Key facts • COVID-19 is a disease caused by the SARS-CoV-2 coronavirus. There are a range of signs and symptoms, but the most common symptoms reported for currently-circulating variants are fever, chills, and sore throat. • Most people make a full recovery without needing treatment. People with severe disease or at risk of developing severe disease should seek medical care as soon as possible. • Nearly 780 million cases and more than 7.1 million deaths due to COVID-19 have been reported worldwide since December 2019, but the actual number is thought to be higher. • Over 13.64 billion COVID-19 vaccine doses had been administered by the end of 2024. WHO continues to recommend COVID-19 vaccination using a risk-based approach. WHO recommends a single dose for persons who have not yet received a COVID-19 vaccine. For those at high risk of severe COVID-19, revaccination may be needed 6-12 months after the most recent dose. • COVID-19 continues to be a significant public health concern, with global circulation of the SARS-CoV-2 virus and hospitalizations and deaths continuing to occur. Overview COVID-19 is caused by the SARS-CoV-2 virus, which spreads through the air in infectious respiratory particles from an infected person. Transmission is more likely in close contact or shared indoor spaces. The virus can also spread by touching contaminated surfaces and then the eyes, nose or mouth. Because of increasing immunity from past infections and/or vaccination, most people recover without treatment. About 6% develop post COVID-19 condition (PCC, or long COVID), however, which can cause lasting symptoms such as memory issues, fatigue or breathing difficulties. PCC can affect daily life and requires individualized care. Older adults, people with underlying health conditions, and unvaccinated individuals are at higher risk of severe illness. Conditions that increase risk include diabetes, heart or lung disease, obesity and weakened immune systems.",dbfs:/Volumes/main/covid_rag/covid_docs/Doc_1_COVID-19_Overview.pdf
Doc_1_COVID-19_Overview.pdf,2,"2 Health and care workers face higher exposure risk due to their work environment. Vaccination remains a key preventive measure against severe disease. COVID-19 vaccines continue to provide strong protection against severe illness and death. Symptoms People may experience different symptoms with COVID-19. The most common symptoms are: • fever • chills • sore throat. Less common symptoms are: • muscle aches and heavy arms or legs • severe fatigue or tiredness • runny or blocked nose, or sneezing • headache • sore eyes • dizziness • new and persistent cough • tight chest or chest pain • shortness of breath • hoarse voice • numbness or tingling • appetite loss, nausea, vomiting, abdominal pain or diarrhoea • loss or change of sense of taste or smell • difficulty sleeping. Immediate medical attention should be sought by people at risk of developing severe disease or with the following severe symptoms: • difficulty breathing, especially at rest, or unable to speak in sentences • confusion • drowsiness or loss of consciousness • persistent pain or pressure in the chest • skin being cold or clammy, or turning pale or a bluish colour",dbfs:/Volumes/main/covid_rag/covid_docs/Doc_1_COVID-19_Overview.pdf
Doc_1_COVID-19_Overview.pdf,3,"3 • loss of speech or movement. People who have underlying health conditions are at higher risk of developing severe disease when they have COVID-19; they should seek medical help early if worried about their condition. These include: • people with high blood pressure, diabetes, obesity, chronic lung, heart, liver and/or kidney diseases, cancer or dementia; • people with rheumatological problems; • women who are pregnant; and • people with immunosuppression, such as those taking immunosuppressive medication or living with HIV. The potential consequences of severe COVID-19 includ

In [0]:
# ============================================================================
# SAVE TO UNITY CATALOG TABLE
# ============================================================================
# Convert pandas DataFrame to Spark DataFrame and save to Unity Catalog
# This makes the data available for downstream chunking and embedding steps
spark.createDataFrame(df_pages).write.mode("overwrite").saveAsTable("main.covid_rag.covid_pages")